## PACOTES ## 

In [1]:
import io
import base64
import warnings
import html
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from scipy import stats

warnings.filterwarnings("ignore")


## CÓDIGO ##

In [ ]:

ARQUIVO_DADOS = "creditcard.csv"
PASTA_SAIDA = "."
ARQUIVO_HTML_SAIDA = "features_analise_exploratoria.html"
PASTA_LATEX = "features_analise_exploratoria"

def formatar_int(valor):
    try:
        return f"{int(valor):,}".replace(",", ".")
    except Exception:
        return str(valor)


def formatar_float(valor, casas=6):
    if pd.isna(valor):
        return "-"

    try:
        return f"{float(valor):.{casas}f}".replace(".", ",")
    except Exception:
        return str(valor)


def formatar_float_latex(valor, casas=6):
    if pd.isna(valor):
        return "-"

    try:
        return f"{float(valor):.{casas}f}"
    except Exception:
        return str(valor)


def formatar_pvalor(valor):
    if pd.isna(valor):
        return "-"

    try:
        valor = float(valor)
    except Exception:
        return str(valor)

    if valor == 0:
        return "< 1e-300"

    return f"{valor:.6e}"


def formatar_pvalor_latex(valor):
    if pd.isna(valor):
        return "-"

    try:
        valor = float(valor)
    except Exception:
        return str(valor)

    if valor == 0:
        return r"$< 10^{-300}$"

    mantissa, expoente = f"{valor:.3e}".split("e")
    expoente = int(expoente)

    return rf"${mantissa} \times 10^{{{expoente}}}$"


def detectar_target(df):
    if "status_fraude" in df.columns:
        return "status_fraude"

    if "Class" in df.columns:
        return "Class"

    return None


def renomear_variavel(nome):
    mapa = {
        "valor_de_transacao": "Valor da transação",
        "tempo_desde_a_primeira_transacao": "Tempo desde a primeira transação"
    }

    return mapa.get(nome, nome)


def decisao_normalidade(pvalor, alpha):
    if pd.isna(pvalor):
        return "Indefinido"

    if pvalor < alpha:
        return "Rejeita normalidade"

    return "Não rejeita normalidade"


def teste_normalidade_jarque_bera_base_inteira(x):

    x = pd.Series(x).dropna().astype(float)

    if len(x) < 3:
        return np.nan, np.nan

    desvio = x.std(ddof=1)

    if desvio == 0 or pd.isna(desvio):
        return np.nan, np.nan

    resultado = stats.jarque_bera(x)

    estatistica = resultado.statistic
    pvalor = resultado.pvalue

    return float(estatistica), float(pvalor)


def fig_to_base64(fig):
    buffer = io.BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=150,
        bbox_inches="tight",
        pad_inches=0.35
    )

    buffer.seek(0)

    return base64.b64encode(buffer.read()).decode("utf-8")


def gerar_tabela_html(df, table_id, classe="data-table"):
    if df is None or df.empty:
        return "<p>Nenhum dado disponível.</p>"

    html_tabela = f'<table id="{table_id}" class="{classe}">\n'
    html_tabela += "<thead><tr>"

    for col in df.columns:
        html_tabela += f"<th>{html.escape(str(col))}</th>"

    html_tabela += "</tr></thead>\n<tbody>\n"

    for _, row in df.iterrows():
        html_tabela += "<tr>"

        for valor in row:
            html_tabela += f"<td>{html.escape(str(valor))}</td>"

        html_tabela += "</tr>\n"

    html_tabela += "</tbody></table>"

    return html_tabela


def preparar_df_formatado_html(df):
    df_fmt = df.copy()

    for col in df_fmt.columns:
        if col in ["P_Valor", "P-valor"]:
            df_fmt[col] = df_fmt[col].apply(formatar_pvalor)

        elif pd.api.types.is_float_dtype(df_fmt[col]):
            df_fmt[col] = df_fmt[col].apply(lambda x: formatar_float(x, 6))

        elif pd.api.types.is_integer_dtype(df_fmt[col]):
            df_fmt[col] = df_fmt[col].apply(formatar_int)

    return df_fmt


def preparar_df_formatado_latex(df):
    df_fmt = df.copy()

    for col in df_fmt.columns:
        if col == "P_Valor":
            df_fmt[col] = df_fmt[col].apply(formatar_pvalor_latex)

        elif pd.api.types.is_float_dtype(df_fmt[col]):
            df_fmt[col] = df_fmt[col].apply(lambda x: formatar_float_latex(x, 6))

        elif pd.api.types.is_integer_dtype(df_fmt[col]):
            df_fmt[col] = df_fmt[col].apply(lambda x: int(x) if not pd.isna(x) else x)

    return df_fmt


def renomear_colunas_latex(df):
    df_export = df.copy()

    nomes_colunas = {
        "Variavel": "Variável",
        "Minimo": "Mínimo",
        "Maximo": "Máximo",
        "Amplitude": "Amplitude",
        "Media": "Média",
        "Mediana": "Mediana",
        "Desvio_Padrao": "Desvio Padrão",
        "Assimetria": "Assimetria",
        "Curtose": "Curtose",
        "P25": "P25",
        "P75": "P75",
        "IQR": "IQR",
        "Limite_Inferior_IQR": "Limite Inferior IQR",
        "Limite_Superior_IQR": "Limite Superior IQR",
        "Outliers_IQR_Total": "Outliers IQR Total",
        "Outliers_IQR_Inferiores": "Outliers IQR Inferiores",
        "Outliers_IQR_Superiores": "Outliers IQR Superiores",
        "Percentual_Outliers_IQR": "Percentual Outliers IQR",
        "Teste_Normalidade": "Teste de Normalidade",
        "JB_Estatistica": "Estatística JB",
        "P_Valor": "P-valor",
        "Decisao_Normalidade_Alpha_0_001": "Decisão Normalidade Alpha 0,001",
        "Decisao_Normalidade_Alpha_0_01": "Decisão Normalidade Alpha 0,01",
        "Decisao_Normalidade_Alpha_0_05": "Decisão Normalidade Alpha 0,05",
        "Decisao_Normalidade_Alpha_0_10": "Decisão Normalidade Alpha 0,10",
        "Alpha": "Alpha",
        "Rejeitam_Normalidade": "Rejeitam Normalidade",
        "Nao_Rejeitam_Normalidade": "Não Rejeitam Normalidade",
    }

    df_export = df_export.rename(columns=nomes_colunas)

    if "Variável" in df_export.columns:
        df_export["Variável"] = df_export["Variável"].apply(renomear_variavel)

    return df_export


def salvar_tabela_latex(df, caminho, caption, label, longtable=True):
    caminho = Path(caminho)

    df_export = renomear_colunas_latex(df)

    tex = df_export.to_latex(
        index=False,
        escape=True,
        longtable=longtable,
        caption=caption,
        label=label
    )

    tex = tex.replace("Continued on next page", "Continua na próxima página")
    tex = tex.replace("Continued from previous page", "Continuação da página anterior")

    caminho.write_text(tex, encoding="utf-8")

def gerar_grafico_pizza_outliers_iqr_fig(tabela_iqr_outliers):
    if (
        tabela_iqr_outliers is None
        or tabela_iqr_outliers.empty
        or "Outliers_IQR_Total" not in tabela_iqr_outliers.columns
        or "Variavel" not in tabela_iqr_outliers.columns
    ):
        return None

    tabela_base = tabela_iqr_outliers[
        ["Variavel", "Outliers_IQR_Total"]
    ].copy()

    tabela_base["Outliers_IQR_Total"] = pd.to_numeric(
        tabela_base["Outliers_IQR_Total"],
        errors="coerce"
    ).fillna(0)

    tabela_zero = tabela_base[
        tabela_base["Outliers_IQR_Total"] == 0
    ].copy()

    qtd_variaveis_zero = len(tabela_zero)

    variaveis_zero = (
        tabela_zero["Variavel"]
        .astype(str)
        .apply(renomear_variavel)
        .tolist()
    )

    tabela_plot = tabela_base[
        tabela_base["Outliers_IQR_Total"] > 0
    ].copy()

    if tabela_plot.empty:
        return None

    tabela_plot = tabela_plot.sort_values(
        by="Outliers_IQR_Total",
        ascending=False
    )

    labels = (
        tabela_plot["Variavel"]
        .astype(str)
        .apply(renomear_variavel)
        .tolist()
    )

    values = tabela_plot["Outliers_IQR_Total"].astype(float).to_numpy()

    total_outliers = values.sum()

    if total_outliers <= 0:
        return None

    def autopct_func(pct):
        quantidade = int(round(pct / 100.0 * total_outliers))
        return f"{pct:.1f}%\n({quantidade:,})".replace(",", ".")

    fig, ax = plt.subplots(figsize=(16, 10))

    wedges, texts, autotexts = ax.pie(
        values,
        labels=labels,
        autopct=autopct_func,
        startangle=90,
        pctdistance=0.72,
        labeldistance=1.08,
        textprops={
            "fontsize": 9,
            "fontweight": "bold"
        },
        wedgeprops={
            "linewidth": 1.0,
            "edgecolor": "white"
        }
    )

    for text in texts:
        text.set_fontsize(9)
        text.set_fontweight("bold")

    for autotext in autotexts:
        autotext.set_fontsize(8)
        autotext.set_fontweight("bold")
        autotext.set_color("black")
        autotext.set_bbox(
            dict(
                boxstyle="round,pad=0.18",
                facecolor="white",
                edgecolor="#cbd5e1",
                linewidth=0.8,
                alpha=0.90
            )
        )

    ax.set_title(
        "Distribuição dos Outliers IQR por Variáveis Numéricas",
        fontsize=17,
        fontweight="bold",
        pad=18
    )

    ax.text(
        0,
        -1.18,
        f"Total de outliers IQR contabilizados: {int(total_outliers):,}".replace(",", "."),
        ha="center",
        va="center",
        fontsize=12,
        fontweight="bold"
    )

    if qtd_variaveis_zero > 0:
        texto_zero = (
            f"{qtd_variaveis_zero} variável com 0 outliers IQR\n"
            "não aparece no gráfico."
            if qtd_variaveis_zero == 1
            else
            f"{qtd_variaveis_zero} variáveis com 0 outliers IQR\n"
            "não aparecem no gráfico."
        )

        if qtd_variaveis_zero <= 5:
            texto_zero += "\n\n" + "\n".join(variaveis_zero)

        legenda_zero = mpatches.Patch(
            facecolor="#e5e7eb",
            edgecolor="#64748b",
            label=texto_zero
        )

        ax.legend(
            handles=[legenda_zero],
            title="Observação",
            loc="center left",
            bbox_to_anchor=(1.03, 0.50),
            fontsize=10,
            title_fontsize=11,
            frameon=True
        )

    ax.axis("equal")

    plt.tight_layout()

    return fig


def cor_correlacao(valor):
    if pd.isna(valor):
        return "#f8fafc"

    valor = float(valor)

    if valor >= 0.90:
        return "#08306b"
    elif valor >= 0.70:
        return "#08519c"
    elif valor >= 0.50:
        return "#2171b5"
    elif valor >= 0.30:
        return "#6baed6"
    elif valor >= 0.10:
        return "#c6dbef"
    elif valor > -0.10:
        return "#f8fafc"
    elif valor > -0.30:
        return "#fee2e2"
    elif valor > -0.50:
        return "#fecaca"
    elif valor > -0.70:
        return "#f87171"
    elif valor > -0.90:
        return "#dc2626"
    else:
        return "#7f1d1d"


def gerar_escala_correlacao_html():
    return """
    <div class="scale-box">
        <div class="scale-title">Escala de cor da correlação</div>
        <div class="scale-bar"></div>
        <div class="scale-labels">
            <span>-1</span>
            <span>0</span>
            <span>+1</span>
        </div>
    </div>
    """


def gerar_matriz_html(matriz, titulo, table_id):
    if matriz is None or matriz.empty:
        return f"""
        <section class="section">
            <h2>{html.escape(titulo)}</h2>
            <p>Nenhuma matriz disponível.</p>
        </section>
        """

    cols = matriz.columns.tolist()

    tabela = f'<div class="matrix-scroll"><table id="{table_id}" class="corr-table">\n'
    tabela += "<thead><tr><th>Variável</th>"

    for col in cols:
        tabela += f"<th>{html.escape(str(renomear_variavel(col)))}</th>"

    tabela += "</tr></thead><tbody>"

    for idx, row in matriz.iterrows():
        tabela += f"<tr><th>{html.escape(str(renomear_variavel(idx)))}</th>"

        for col in cols:
            val = row[col]
            bg = cor_correlacao(val)
            texto = "" if pd.isna(val) else f"{float(val):.3f}"

            if pd.isna(val):
                color = "#020617"
            else:
                color = "#ffffff" if abs(float(val)) >= 0.70 else "#020617"

            tabela += (
                f'<td style="background:{bg}; color:{color};">'
                f'{texto}</td>'
            )

        tabela += "</tr>"

    tabela += "</tbody></table></div>"

    escala = gerar_escala_correlacao_html()

    return f"""
    <section class="section">
        <div class="section-header">
            <h2>{html.escape(titulo)}</h2>
            <button class="download-btn" onclick="baixarTabelaCSV('{table_id}', '{table_id}.csv')">
                Baixar CSV
            </button>
        </div>
        {escala}
        {tabela}
    </section>
    """


def gerar_figura_matriz_correlacao(matriz, titulo):
    matriz_plot = matriz.copy()

    n_linhas, n_colunas = matriz_plot.shape
    tamanho_figura = max(18, n_colunas * 0.70)

    fig, ax = plt.subplots(
        figsize=(tamanho_figura, tamanho_figura)
    )

    im = ax.imshow(
        matriz_plot.values,
        vmin=-1,
        vmax=1,
        cmap="coolwarm",
        aspect="equal"
    )

    cbar = fig.colorbar(
        im,
        ax=ax,
        fraction=0.046,
        pad=0.04
    )

    cbar.set_label(
        "Correlação",
        fontsize=14,
        fontweight="bold"
    )

    labels_x = [renomear_variavel(x) for x in matriz_plot.columns.tolist()]
    labels_y = [renomear_variavel(x) for x in matriz_plot.index.tolist()]

    ax.set_xticks(np.arange(n_colunas))
    ax.set_yticks(np.arange(n_linhas))

    ax.set_xticklabels(
        labels_x,
        rotation=90,
        ha="center",
        fontsize=8
    )

    ax.set_yticklabels(
        labels_y,
        fontsize=8
    )

    ax.set_xticks(
        np.arange(-0.5, n_colunas, 1),
        minor=True
    )

    ax.set_yticks(
        np.arange(-0.5, n_linhas, 1),
        minor=True
    )

    ax.grid(
        which="minor",
        color="white",
        linestyle="-",
        linewidth=0.9
    )

    ax.tick_params(
        which="minor",
        bottom=False,
        left=False
    )

    for i in range(n_linhas):
        for j in range(n_colunas):
            valor = matriz_plot.iloc[i, j]

            if pd.isna(valor):
                texto = ""
                cor_texto = "black"
            else:
                texto = f"{valor:.2f}"
                cor_texto = "white" if abs(valor) >= 0.60 else "black"

            ax.text(
                j,
                i,
                texto,
                ha="center",
                va="center",
                fontsize=5.5,
                color=cor_texto,
                fontweight="bold"
            )

    ax.set_title(
        titulo,
        fontsize=19,
        fontweight="bold",
        pad=24
    )

    plt.subplots_adjust(
        left=0.24,
        right=0.90,
        top=0.92,
        bottom=0.26
    )

    return fig

def gerar_card(titulo, valor, subtitulo=None):
    subtitulo_html = ""

    if subtitulo is not None:
        subtitulo_html = f'<div class="card-subtitle">{subtitulo}</div>'

    return f"""
    <div class="metric-card">
        <div class="card-title">{titulo}</div>
        <div class="card-value">{valor}</div>
        {subtitulo_html}
    </div>
    """

def exportar_artefatos_latex(
    pasta_latex,
    tabela_estatisticas_basicas,
    tabela_iqr_outliers,
    tabela_normalidade,
    resumo_normalidade,
    matriz_pearson,
    matriz_spearman,
    grafico_outliers_iqr_fig=None
):
    pasta_latex = Path(pasta_latex)
    pasta_latex.mkdir(parents=True, exist_ok=True)

    tabela_estatisticas_basicas_latex = preparar_df_formatado_latex(
        tabela_estatisticas_basicas
    )

    tabela_iqr_outliers_latex = preparar_df_formatado_latex(
        tabela_iqr_outliers
    )

    tabela_normalidade_latex = preparar_df_formatado_latex(
        tabela_normalidade
    )

    resumo_normalidade_latex = preparar_df_formatado_latex(
        resumo_normalidade
    )

    salvar_tabela_latex(
        tabela_estatisticas_basicas_latex,
        pasta_latex / "tabela_estatisticas_basicas_features.tex",
        caption="Estatísticas descritivas básicas das features numéricas.",
        label="tab:estatisticas-basicas-features",
        longtable=True
    )

    salvar_tabela_latex(
        tabela_iqr_outliers_latex,
        pasta_latex / "tabela_iqr_outliers_features.tex",
        caption="Quartis, limites IQR e outliers das features numéricas.",
        label="tab:iqr-outliers-features",
        longtable=True
    )

    salvar_tabela_latex(
        tabela_normalidade_latex,
        pasta_latex / "tabela_normalidade.tex",
        caption="Teste de normalidade Jarque-Bera aplicado às features numéricas.",
        label="tab:normalidade-features",
        longtable=True
    )

    salvar_tabela_latex(
        resumo_normalidade_latex,
        pasta_latex / "tabela_resumo_normalidade.tex",
        caption="Resumo das decisões do teste de normalidade por nível de significância.",
        label="tab:resumo-normalidade",
        longtable=False
    )

    if grafico_outliers_iqr_fig is not None:
        grafico_outliers_iqr_fig.savefig(
            pasta_latex / "grafico_outliers_iqr_por_feature.pdf",
            bbox_inches="tight",
            pad_inches=0.35
        )

        grafico_outliers_iqr_fig.savefig(
            pasta_latex / "grafico_outliers_iqr_por_feature.png",
            dpi=300,
            bbox_inches="tight",
            pad_inches=0.35
        )

    fig_pearson = gerar_figura_matriz_correlacao(
        matriz_pearson,
        "Matriz de Correlação de Pearson"
    )

    fig_pearson.savefig(
        pasta_latex / "matriz_pearson.pdf",
        bbox_inches="tight",
        pad_inches=0.45
    )

    fig_pearson.savefig(
        pasta_latex / "matriz_pearson.png",
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.45
    )

    plt.close(fig_pearson)

    fig_spearman = gerar_figura_matriz_correlacao(
        matriz_spearman,
        "Matriz de Correlação de Spearman"
    )

    fig_spearman.savefig(
        pasta_latex / "matriz_spearman.pdf",
        bbox_inches="tight",
        pad_inches=0.45
    )

    fig_spearman.savefig(
        pasta_latex / "matriz_spearman.png",
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.45
    )

    plt.close(fig_spearman)

    comandos_latex = r"""

% Pacotes sugeridos no preâmbulo:
% \usepackage{graphicx}
% \usepackage{float}
% \usepackage{booktabs}
% \usepackage{longtable}
% \usepackage{pdflscape}

\begin{table}[H]
\centering
\caption{Resumo das decisões do teste de normalidade.}
\label{tab:resumo-normalidade-main}
\input{features_analise_exploratoria/tabela_resumo_normalidade.tex}
\end{table}

\begin{landscape}
\small
\input{features_analise_exploratoria/tabela_estatisticas_basicas_features.tex}
\end{landscape}

\begin{landscape}
\small
\input{features_analise_exploratoria/tabela_iqr_outliers_features.tex}
\end{landscape}

\begin{landscape}
\small
\input{features_analise_exploratoria/tabela_normalidade.tex}
\end{landscape}

\begin{figure}[H]
\centering
\includegraphics[width=0.85\textwidth]{features_analise_exploratoria/grafico_outliers_iqr_por_feature.pdf}
\caption{Distribuição percentual dos outliers detectados pelo critério IQR por feature.}
\label{fig:grafico-outliers-iqr-por-feature}
\end{figure}

\begin{landscape}
\begin{figure}[H]
\centering
\includegraphics[width=1.00\linewidth]{features_analise_exploratoria/matriz_pearson.pdf}
\caption{Matriz de correlação de Pearson entre as features numéricas.}
\label{fig:matriz-pearson}
\end{figure}
\end{landscape}

\begin{landscape}
\begin{figure}[H]
\centering
\includegraphics[width=1.00\linewidth]{features_analise_exploratoria/matriz_spearman.pdf}
\caption{Matriz de correlação de Spearman entre as features numéricas.}
\label{fig:matriz-spearman}
\end{figure}
\end{landscape}
"""

    (pasta_latex / "comandos_latex_exemplo.tex").write_text(
        comandos_latex,
        encoding="utf-8"
    )


def gerar_analise_exploratoria_features(
    arquivo_dados=ARQUIVO_DADOS,
    pasta_saida=PASTA_SAIDA,
    arquivo_html_saida=ARQUIVO_HTML_SAIDA,
    pasta_latex=PASTA_LATEX,
    exportar_latex=True
):
    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)

    caminho_html = pasta_saida / arquivo_html_saida
    caminho_latex = pasta_saida / pasta_latex

    df = pd.read_csv(arquivo_dados)

    target_name = detectar_target(df)

    features_numericas = [
        col for col in df.columns
        if col != target_name
        and pd.api.types.is_numeric_dtype(df[col])
    ]

    if len(features_numericas) == 0:
        raise ValueError("Nenhuma feature numérica encontrada após remover o rótulo.")

    linhas_estatisticas_basicas = []
    linhas_iqr_outliers = []
    linhas_normalidade = []

    for feature in features_numericas:
        x = df[feature].dropna().astype(float)

        minimo = x.min()
        maximo = x.max()
        amplitude = maximo - minimo
        media = x.mean()
        mediana = x.median()
        desvio_padrao = x.std(ddof=1)
        assimetria = x.skew()
        curtose = x.kurtosis()

        p25 = x.quantile(0.25)
        p75 = x.quantile(0.75)
        iqr = p75 - p25

        limite_inferior_iqr = p25 - 1.5 * iqr
        limite_superior_iqr = p75 + 1.5 * iqr

        outliers_iqr_inferiores = int((x < limite_inferior_iqr).sum())
        outliers_iqr_superiores = int((x > limite_superior_iqr).sum())
        outliers_iqr_total = outliers_iqr_inferiores + outliers_iqr_superiores

        percentual_outliers_iqr = (
            outliers_iqr_total / len(x) * 100
            if len(x) > 0
            else np.nan
        )

        jb_estatistica, pvalor = teste_normalidade_jarque_bera_base_inteira(x)

        linhas_estatisticas_basicas.append({
            "Variavel": feature,
            "Minimo": minimo,
            "Maximo": maximo,
            "Amplitude": amplitude,
            "Media": media,
            "Mediana": mediana,
            "Desvio_Padrao": desvio_padrao,
            "Assimetria": assimetria,
            "Curtose": curtose,
        })

        linhas_iqr_outliers.append({
            "Variavel": feature,
            "P25": p25,
            "P75": p75,
            "IQR": iqr,
            "Limite_Inferior_IQR": limite_inferior_iqr,
            "Limite_Superior_IQR": limite_superior_iqr,
            "Outliers_IQR_Total": outliers_iqr_total,
            "Outliers_IQR_Inferiores": outliers_iqr_inferiores,
            "Outliers_IQR_Superiores": outliers_iqr_superiores,
            "Percentual_Outliers_IQR": percentual_outliers_iqr,
        })

        linhas_normalidade.append({
            "Variavel": feature,
            "Teste_Normalidade": "Jarque-Bera",
            "JB_Estatistica": jb_estatistica,
            "P_Valor": pvalor,
            "Decisao_Normalidade_Alpha_0_001": decisao_normalidade(pvalor, 0.001),
            "Decisao_Normalidade_Alpha_0_01": decisao_normalidade(pvalor, 0.01),
            "Decisao_Normalidade_Alpha_0_05": decisao_normalidade(pvalor, 0.05),
            "Decisao_Normalidade_Alpha_0_10": decisao_normalidade(pvalor, 0.10),
        })

    tabela_estatisticas_basicas = pd.DataFrame(linhas_estatisticas_basicas)
    tabela_iqr_outliers = pd.DataFrame(linhas_iqr_outliers)
    tabela_normalidade = pd.DataFrame(linhas_normalidade)

    colunas_estatisticas_basicas = [
        "Variavel",
        "Minimo",
        "Maximo",
        "Amplitude",
        "Media",
        "Mediana",
        "Desvio_Padrao",
        "Assimetria",
        "Curtose",
    ]

    colunas_iqr_outliers = [
        "Variavel",
        "P25",
        "P75",
        "IQR",
        "Limite_Inferior_IQR",
        "Limite_Superior_IQR",
        "Outliers_IQR_Total",
        "Outliers_IQR_Inferiores",
        "Outliers_IQR_Superiores",
        "Percentual_Outliers_IQR",
    ]

    colunas_normalidade = [
        "Variavel",
        "Teste_Normalidade",
        "JB_Estatistica",
        "P_Valor",
        "Decisao_Normalidade_Alpha_0_001",
        "Decisao_Normalidade_Alpha_0_01",
        "Decisao_Normalidade_Alpha_0_05",
        "Decisao_Normalidade_Alpha_0_10",
    ]

    tabela_estatisticas_basicas = tabela_estatisticas_basicas[colunas_estatisticas_basicas]
    tabela_iqr_outliers = tabela_iqr_outliers[colunas_iqr_outliers]
    tabela_normalidade = tabela_normalidade[colunas_normalidade]

    df_features = df[features_numericas].copy()

    matriz_pearson = df_features.corr(method="pearson")
    matriz_spearman = df_features.corr(method="spearman")

    grafico_outliers_iqr_fig = gerar_grafico_pizza_outliers_iqr_fig(
        tabela_iqr_outliers
    )

    if grafico_outliers_iqr_fig is not None:
        grafico_outliers_iqr = fig_to_base64(grafico_outliers_iqr_fig)
    else:
        grafico_outliers_iqr = None

    total_features = len(features_numericas)

    rejeitam_0001 = int((tabela_normalidade["P_Valor"] < 0.001).sum())
    rejeitam_001 = int((tabela_normalidade["P_Valor"] < 0.01).sum())
    rejeitam_005 = int((tabela_normalidade["P_Valor"] < 0.05).sum())
    rejeitam_010 = int((tabela_normalidade["P_Valor"] < 0.10).sum())

    nao_rejeitam_0001 = int((tabela_normalidade["P_Valor"] >= 0.001).sum())
    nao_rejeitam_001 = int((tabela_normalidade["P_Valor"] >= 0.01).sum())
    nao_rejeitam_005 = int((tabela_normalidade["P_Valor"] >= 0.05).sum())
    nao_rejeitam_010 = int((tabela_normalidade["P_Valor"] >= 0.10).sum())

    resumo_normalidade = pd.DataFrame([
        {
            "Alpha": "0.001",
            "Rejeitam_Normalidade": rejeitam_0001,
            "Nao_Rejeitam_Normalidade": nao_rejeitam_0001
        },
        {
            "Alpha": "0.01",
            "Rejeitam_Normalidade": rejeitam_001,
            "Nao_Rejeitam_Normalidade": nao_rejeitam_001
        },
        {
            "Alpha": "0.05",
            "Rejeitam_Normalidade": rejeitam_005,
            "Nao_Rejeitam_Normalidade": nao_rejeitam_005
        },
        {
            "Alpha": "0.10",
            "Rejeitam_Normalidade": rejeitam_010,
            "Nao_Rejeitam_Normalidade": nao_rejeitam_010
        }
    ])

    if exportar_latex:
        exportar_artefatos_latex(
            pasta_latex=caminho_latex,
            tabela_estatisticas_basicas=tabela_estatisticas_basicas,
            tabela_iqr_outliers=tabela_iqr_outliers,
            tabela_normalidade=tabela_normalidade,
            resumo_normalidade=resumo_normalidade,
            matriz_pearson=matriz_pearson,
            matriz_spearman=matriz_spearman,
            grafico_outliers_iqr_fig=grafico_outliers_iqr_fig
        )

    if grafico_outliers_iqr_fig is not None:
        plt.close(grafico_outliers_iqr_fig)

    tabela_estatisticas_basicas_fmt = preparar_df_formatado_html(
        renomear_colunas_latex(tabela_estatisticas_basicas)
    )

    tabela_iqr_outliers_fmt = preparar_df_formatado_html(
        renomear_colunas_latex(tabela_iqr_outliers)
    )

    tabela_normalidade_fmt = preparar_df_formatado_html(
        renomear_colunas_latex(tabela_normalidade)
    )

    tabela_estatisticas_basicas_html = gerar_tabela_html(
        tabela_estatisticas_basicas_fmt,
        table_id="tabela_estatisticas_basicas"
    )

    tabela_iqr_outliers_html = gerar_tabela_html(
        tabela_iqr_outliers_fmt,
        table_id="tabela_iqr_outliers"
    )

    tabela_normalidade_html = gerar_tabela_html(
        tabela_normalidade_fmt,
        table_id="tabela_normalidade"
    )

    if grafico_outliers_iqr is not None:
        grafico_outliers_iqr_html = f"""
        <section class="section">
            <div class="section-actions-only">
                <a
                    class="download-btn link-btn"
                    href="data:image/png;base64,{grafico_outliers_iqr}"
                    download="grafico_outliers_iqr_por_feature.png"
                >
                    Baixar PNG
                </a>
            </div>
            <img
                id="grafico_outliers_iqr"
                class="plot-img"
                src="data:image/png;base64,{grafico_outliers_iqr}"
                alt="Distribuição dos Outliers IQR por Variáveis Numéricas"
            >
        </section>
        """
    else:
        grafico_outliers_iqr_html = ""

    matriz_pearson_html = gerar_matriz_html(
        matriz_pearson,
        "Matriz de Correlação de Pearson",
        table_id="matriz_pearson"
    )

    matriz_spearman_html = gerar_matriz_html(
        matriz_spearman,
        "Matriz de Correlação de Spearman",
        table_id="matriz_spearman"
    )

    html_final = f"""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <title>Análise Exploratória das Features</title>

        <style>
            body {{
                font-family: Arial, Helvetica, sans-serif;
                background: #f4f6f8;
                color: #020617;
                margin: 0;
                padding: 32px;
            }}

            .container {{
                max-width: 1450px;
                margin: 0 auto;
            }}

            h1 {{
                text-align: center;
                margin-bottom: 34px;
                color: #020617;
                font-size: 34px;
            }}

            .section {{
                background: white;
                border-radius: 16px;
                padding: 24px;
                margin-bottom: 28px;
                box-shadow: 0 8px 24px rgba(15, 23, 42, 0.08);
            }}

            .section h2 {{
                text-align: center;
                margin-top: 0;
                margin-bottom: 22px;
                font-size: 24px;
                color: #020617;
            }}

            .section-header {{
                display: flex;
                align-items: center;
                justify-content: center;
                gap: 14px;
                flex-wrap: wrap;
                margin-bottom: 18px;
            }}

            .section-header h2 {{
                margin: 0;
            }}

            .section-actions-only {{
                display: flex;
                justify-content: flex-end;
                margin-bottom: 12px;
            }}

            .download-btn {{
                border: 1px solid #bfdbfe;
                background: #eff6ff;
                color: #1e3a8a;
                padding: 8px 12px;
                border-radius: 10px;
                font-weight: 800;
                font-size: 13px;
                cursor: pointer;
                text-decoration: none;
                display: inline-block;
            }}

            .download-btn:hover {{
                background: #dbeafe;
            }}

            .metric-grid {{
                display: grid;
                grid-template-columns: repeat(5, 1fr);
                gap: 14px;
                margin-top: 18px;
            }}

            .metric-card {{
                background: #f8fafc;
                border: 1px solid #e2e8f0;
                border-radius: 14px;
                padding: 16px;
            }}

            .card-title {{
                font-size: 13px;
                font-weight: 800;
                color: #475569;
                margin-bottom: 8px;
            }}

            .card-value {{
                font-size: 22px;
                font-weight: 900;
                color: #020617;
                font-family: Consolas, Monaco, monospace;
            }}

            .card-subtitle {{
                font-size: 12px;
                color: #64748b;
                margin-top: 8px;
                line-height: 1.35;
            }}

            .table-wrapper {{
                overflow-x: auto;
                border: 1px solid #e2e8f0;
                border-radius: 12px;
                max-height: 620px;
                overflow-y: auto;
            }}

            .data-table {{
                width: 100%;
                border-collapse: collapse;
                font-size: 13px;
                margin-top: 0;
            }}

            .data-table th {{
                background: #0f172a;
                color: white;
                padding: 10px 8px;
                text-align: left;
                position: sticky;
                top: 0;
                z-index: 1;
            }}

            .data-table td {{
                border-bottom: 1px solid #e2e8f0;
                padding: 8px;
                color: #020617;
                white-space: nowrap;
            }}

            .data-table tr:nth-child(even) {{
                background: #f8fafc;
            }}

            .matrix-scroll {{
                overflow: auto;
                border: 1px solid #e2e8f0;
                border-radius: 12px;
                max-height: 720px;
            }}

            .corr-table {{
                border-collapse: collapse;
                font-size: 11px;
                min-width: 100%;
            }}

            .corr-table th {{
                background: #0f172a;
                color: white;
                padding: 7px;
                position: sticky;
                top: 0;
                z-index: 2;
            }}

            .corr-table th:first-child {{
                left: 0;
                z-index: 3;
            }}

            .corr-table td {{
                text-align: center;
                padding: 6px;
                border: 1px solid #e2e8f0;
                font-weight: 700;
                min-width: 58px;
            }}

            .scale-box {{
                max-width: 560px;
                margin: 0 auto 18px auto;
            }}

            .scale-title {{
                text-align: center;
                font-size: 13px;
                font-weight: 800;
                color: #334155;
                margin-bottom: 6px;
            }}

            .scale-bar {{
                height: 18px;
                border-radius: 999px;
                border: 1px solid #cbd5e1;
                background: linear-gradient(
                    to right,
                    #7f1d1d 0%,
                    #dc2626 15%,
                    #f87171 30%,
                    #fee2e2 42%,
                    #f8fafc 50%,
                    #c6dbef 58%,
                    #6baed6 70%,
                    #2171b5 82%,
                    #08519c 92%,
                    #08306b 100%
                );
            }}

            .scale-labels {{
                display: flex;
                justify-content: space-between;
                font-size: 12px;
                font-weight: 700;
                color: #475569;
                margin-top: 5px;
            }}

            .plot-img {{
                display: block;
                max-width: 100%;
                margin: 0 auto;
                border-radius: 12px;
                border: 1px solid #e2e8f0;
            }}

            .latex-box {{
                background: #f8fafc;
                border: 1px solid #e2e8f0;
                border-radius: 14px;
                padding: 16px;
                color: #334155;
                line-height: 1.55;
                font-size: 14px;
            }}

            @media (max-width: 1200px) {{
                .metric-grid {{
                    grid-template-columns: repeat(2, 1fr);
                }}
            }}

            @media (max-width: 640px) {{
                .metric-grid {{
                    grid-template-columns: 1fr;
                }}
            }}
        </style>
    </head>

    <body>
        <div class="container">

            <h1>Análise Exploratória das Features Numéricas</h1>

            <section class="section">
                <h2>Resumo Geral</h2>

                <div class="metric-grid">
                    {gerar_card("Features numéricas", formatar_int(total_features), "Variável de rótulo removida da análise.")}
                    {gerar_card("Teste de normalidade", "Jarque-Bera", "H0: assimetria e curtose compatíveis com normalidade.")}
                    {gerar_card("Rejeitam α = 0.001", formatar_int(rejeitam_0001), "p-valor < 0.001.")}
                    {gerar_card("Rejeitam α = 0.01", formatar_int(rejeitam_001), "p-valor < 0.01.")}
                    {gerar_card("Rejeitam α = 0.05", formatar_int(rejeitam_005), "p-valor < 0.05.")}
                    {gerar_card("Rejeitam α = 0.10", formatar_int(rejeitam_010), "p-valor < 0.10.")}
                    {gerar_card("Não rejeitam α = 0.001", formatar_int(nao_rejeitam_0001), "p-valor >= 0.001.")}
                    {gerar_card("Não rejeitam α = 0.01", formatar_int(nao_rejeitam_001), "p-valor >= 0.01.")}
                    {gerar_card("Não rejeitam α = 0.05", formatar_int(nao_rejeitam_005), "p-valor >= 0.05.")}
                    {gerar_card("Não rejeitam α = 0.10", formatar_int(nao_rejeitam_010), "p-valor >= 0.10.")}
                </div>
            </section>

            <section class="section">
                <h2>Critério do IQR para Detecção de Outliers</h2>
                <div class="latex-box">
                    O critério do IQR, também conhecido como critério de Tukey,
                    identifica como outliers os valores localizados abaixo de
                    <strong>Q1 - 1,5 × IQR</strong> ou acima de
                    <strong>Q3 + 1,5 × IQR</strong>, em que
                    <strong>IQR = Q3 - Q1</strong>. Assim, para cada variável,
                    o relatório contabiliza o número total de outliers, os outliers
                    inferiores e os outliers superiores.
                </div>
            </section>

            {grafico_outliers_iqr_html}

            <section class="section">
                <div class="section-header">
                    <h2>Estatísticas Descritivas Básicas das Features</h2>
                    <button class="download-btn" onclick="baixarTabelaCSV('tabela_estatisticas_basicas', 'tabela_estatisticas_basicas.csv')">
                        Baixar CSV
                    </button>
                </div>
                <div class="table-wrapper">
                    {tabela_estatisticas_basicas_html}
                </div>
            </section>

            <section class="section">
                <div class="section-header">
                    <h2>Quartis, IQR e Outliers das Features</h2>
                    <button class="download-btn" onclick="baixarTabelaCSV('tabela_iqr_outliers', 'tabela_iqr_outliers.csv')">
                        Baixar CSV
                    </button>
                </div>
                <div class="table-wrapper">
                    {tabela_iqr_outliers_html}
                </div>
            </section>

            <section class="section">
                <div class="section-header">
                    <h2>Tabela de Normalidade das Features</h2>
                    <button class="download-btn" onclick="baixarTabelaCSV('tabela_normalidade', 'tabela_normalidade.csv')">
                        Baixar CSV
                    </button>
                </div>
                <div class="table-wrapper">
                    {tabela_normalidade_html}
                </div>
            </section>

            {matriz_pearson_html}

            {matriz_spearman_html}

        </div>

        <script>
            function limparTextoCSV(texto) {{
                if (texto === null || texto === undefined) {{
                    return "";
                }}

                texto = String(texto)
                    .replace(/[\\r\\n]+/g, " ")
                    .replace(/[\\s]+/g, " ")
                    .trim();

                if (texto.includes(";") || texto.includes('"')) {{
                    texto = '"' + texto.replace(/"/g, '""') + '"';
                }}

                return texto;
            }}

            function baixarTabelaCSV(tableId, filename) {{
                const tabela = document.getElementById(tableId);

                if (!tabela) {{
                    alert("Tabela não encontrada: " + tableId);
                    return;
                }}

                const linhas = [];

                tabela.querySelectorAll("tr").forEach(function(row) {{
                    const celulas = Array.from(row.querySelectorAll("th, td"));
                    const linha = celulas
                        .map(celula => limparTextoCSV(celula.innerText))
                        .join(";");

                    linhas.push(linha);
                }});

                const csv = "\\ufeff" + linhas.join("\\n");
                const blob = new Blob([csv], {{ type: "text/csv;charset=utf-8;" }});
                const url = URL.createObjectURL(blob);

                const link = document.createElement("a");
                link.href = url;
                link.download = filename;
                document.body.appendChild(link);
                link.click();
                document.body.removeChild(link);

                URL.revokeObjectURL(url);
            }}
        </script>
    </body>
    </html>
    """

    caminho_html.write_text(
        html_final,
        encoding="utf-8"
    )

    print("=" * 80)
    print("ANÁLISE EXPLORATÓRIA FINALIZADA")
    print("=" * 80)
    print(f"HTML salvo em: {caminho_html.resolve()}")

    if exportar_latex:
        print(f"Arquivos LaTeX salvos em: {caminho_latex.resolve()}")

    print(f"Features numéricas analisadas: {total_features}")
    print("=" * 80)

    return {
        "tabela_estatisticas_basicas": tabela_estatisticas_basicas,
        "tabela_iqr_outliers": tabela_iqr_outliers,
        "tabela_normalidade": tabela_normalidade,
        "resumo_normalidade": resumo_normalidade,
        "matriz_pearson": matriz_pearson,
        "matriz_spearman": matriz_spearman,
        "caminho_html": caminho_html,
        "caminho_latex": caminho_latex
    }


resultado_eda_features = gerar_analise_exploratoria_features(
    arquivo_dados="creditcard.csv",
    pasta_saida=".",
    arquivo_html_saida="features_analise_exploratoria.html",
    pasta_latex="features_analise_exploratoria",
    exportar_latex=True
)

resultado_eda_features["caminho_html"]

ANÁLISE EXPLORATÓRIA FINALIZADA
HTML salvo em: C:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\features_analise_exploratoria.html
Arquivos LaTeX salvos em: C:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\features_analise_exploratoria
Features numéricas analisadas: 30


WindowsPath('features_analise_exploratoria.html')